# 01 — Heuristic Labeling

Loads raw NFT transfer data from the Simiotic Ethereum NFT dataset and applies four heuristic rules to generate weak wash trading labels.

**Input:** `nfts.sqlite` (raw Simiotic dataset)  
**Output:** `labeled_transactions_april_sept.csv`

---

### Labeling Rules

| Rule | Description | Source |
|------|-------------|--------|
| Rule 1 | Seller repurchases the same token within 30 days | Von Wachter et al. (2022) |
| Rule 2 | Sender and receiver share the same address (self-trade) | La Morgia et al. (2023) |
| Rule 3 | Closed loop A → B → ··· → A on one token within 7 days | Von Wachter et al. (2022) |
| Rule 4 | Directed wallet pair exchanges the same token 3 or more times | Niu et al. (2024) |

A transaction is labeled as wash trading if it satisfies **at least one** rule. This recall-focused approach prefers over-identification over missing real manipulation.

---

### Dataset window
1 April 2021 – 25 September 2021 (peak NFT trading activity)

## 1. Load and preprocess raw transfers

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# Connect to the Simiotic raw dataset
# Download from: https://www.kaggle.com/datasets/simiotic/ethereum-nfts
conn = sqlite3.connect('../data/nfts.sqlite/nfts.sqlite')

print("Loading raw transfers (April - September 2021)...")

APRIL_1 = 1617235200   # 2021-04-01
OCT_1   = 1633046400   # 2021-10-01

transfers_raw = pd.read_sql_query(f"""
    SELECT
        transaction_hash,
        timestamp,
        nft_address,
        token_id,
        from_address,
        to_address,
        transaction_value
    FROM transfers
    WHERE timestamp >= {APRIL_1}
      AND timestamp <  {OCT_1}
""", conn)

print(f"Loaded: {len(transfers_raw):,} rows")

# Fix types
transfers_raw['timestamp_dt'] = pd.to_datetime(
    transfers_raw['timestamp'], unit='s'
)
transfers_raw['transaction_value'] = pd.to_numeric(
    transfers_raw['transaction_value'], errors='coerce'
).fillna(0.0)

# Keep only genuine sales (value > 0)
sales = transfers_raw[
    transfers_raw['transaction_value'] > 0
].copy()
print(f"Sales only (value > 0): {len(sales):,}")

# Remove burn addresses (minting and destruction events)
BURN = {
    '0x0000000000000000000000000000000000000000',
    '0x000000000000000000000000000000000000dead'
}
burn_lower = {a.lower() for a in BURN}

sales_clean = sales[
    ~sales['from_address'].str.lower().isin(burn_lower) &
    ~sales['to_address'].str.lower().isin(burn_lower)
].copy().reset_index(drop=True)

print(f"After burn filter: {len(sales_clean):,}")

# Sort chronologically — required for temporal rules
sales_clean = sales_clean.sort_values(
    'timestamp'
).reset_index(drop=True)

## 2. Apply heuristic labeling rules (Rules 1–4)

In [ ]:
# Initialize
sales_clean['is_wash_trading'] = 0
wash_hashes = set()

# Thresholds
MAX_SECS_BUYBACK  = 30 * 24 * 3600  # Rule 1: 30 days
MAX_SECS_CYCLE    = 7  * 24 * 3600  # Rule 3: 7 days
MAX_HOPS          = 10              # Rule 3: max chain hops
PAIR_TX_THRESHOLD = 3               # Rule 4: min tx count per pair

grouped      = sales_clean.groupby(['nft_address', 'token_id'])
total_groups = len(grouped)

print(f"Unique NFT tokens: {total_groups:,}")
print(f"Processing Rules 1, 2, 3...")

r1_count = 0
r2_count = 0
r3_count = 0
processed = 0

for (nft_addr, token_id), group in grouped:

    if len(group) < 1:
        processed += 1
        continue

    group = group.sort_values('timestamp').reset_index(drop=True)
    n = len(group)

    for i in range(n):
        from_i = group.loc[i, 'from_address'].lower()
        to_i   = group.loc[i, 'to_address'].lower()
        hash_i = group.loc[i, 'transaction_hash']
        ts_i   = group.loc[i, 'timestamp']

        # Rule 2: Identity trade (self-trade)
        if from_i == to_i:
            wash_hashes.add(hash_i)
            r2_count += 1
            continue

        chain_hashes = [hash_i]
        chain_end    = to_i

        for j in range(i + 1, n):
            ts_j      = group.loc[j, 'timestamp']
            diff_secs = ts_j - ts_i

            # Rule 1: Seller buyback within 30 days
            if diff_secs > MAX_SECS_BUYBACK:
                break

            from_j = group.loc[j, 'from_address'].lower()
            to_j   = group.loc[j, 'to_address'].lower()
            hash_j = group.loc[j, 'transaction_hash']

            if from_j == chain_end:
                chain_hashes.append(hash_j)
                chain_end = to_j

                # Rule 1: original seller buys back
                if to_j == from_i and diff_secs <= MAX_SECS_BUYBACK:
                    for h in chain_hashes:
                        wash_hashes.add(h)
                    r1_count += 1
                    break

                # Rule 3: closed loop within 7 days
                if (to_j == from_i
                        and diff_secs <= MAX_SECS_CYCLE
                        and len(chain_hashes) <= MAX_HOPS):
                    for h in chain_hashes:
                        wash_hashes.add(h)
                    r3_count += 1
                    break

    processed += 1
    if processed % 100000 == 0:
        pct = processed / total_groups * 100
        print(f"  Progress: {processed:,} / {total_groups:,} ({pct:.1f}%)")

print(f"\nRule 1 (seller buyback)  : {r1_count:,} pairs")
print(f"Rule 2 (identity trade)  : {r2_count:,} transactions")
print(f"Rule 3 (multi-hop cycle) : {r3_count:,} cycles")
print(f"Total hashes so far      : {len(wash_hashes):,}")

# Rule 4: High transaction count per directed wallet pair
print(f"\nProcessing Rule 4 (high tx count per wallet pair)...")

pair_counts = sales_clean.groupby(
    ['from_address', 'to_address', 'nft_address', 'token_id']
).size().reset_index(name='count')

suspicious_pairs = pair_counts[
    pair_counts['count'] >= PAIR_TX_THRESHOLD
]

print(f"  Suspicious wallet pairs: {len(suspicious_pairs):,}")

r4_hashes = set()
for _, row in suspicious_pairs.iterrows():
    mask = (
        (sales_clean['from_address'] == row['from_address']) &
        (sales_clean['to_address'] == row['to_address']) &
        (sales_clean['nft_address'] == row['nft_address']) &
        (sales_clean['token_id'] == row['token_id'])
    )
    for h in sales_clean.loc[mask, 'transaction_hash']:
        if h not in wash_hashes:
            r4_hashes.add(h)

wash_hashes.update(r4_hashes)
print(f"  New hashes from Rule 4 : {len(r4_hashes):,}")

# Apply labels
sales_clean['is_wash_trading'] = sales_clean[
    'transaction_hash'
].isin(wash_hashes).astype(int)

## 3. Labeling summary and sanity check

In [ ]:
print("=" * 55)
print("LABELING RESULTS")
print("=" * 55)
print(f"Total sales      : {len(sales_clean):,}")
print(f"Wash trading (1) : {sales_clean['is_wash_trading'].sum():,} "
      f"({sales_clean['is_wash_trading'].mean():.3%})")
print(f"Normal (0)       : {(sales_clean['is_wash_trading']==0).sum():,}")

# Sanity check: no burn addresses in labeled transactions
burn_in_labels = sales_clean[
    sales_clean['is_wash_trading'] == 1
][['from_address', 'to_address']].apply(
    lambda col: col.str.lower().isin(burn_lower)
).any().any()

print(f"\nSanity check — burn addresses in labels: {burn_in_labels}")
print("(Expected: False)")

## 4. Save output

In [ ]:
output_cols = [
    'transaction_hash',
    'timestamp',
    'nft_address',
    'token_id',
    'from_address',
    'to_address',
    'transaction_value',
    'is_wash_trading'
]

sales_clean[output_cols].to_csv(
    'labeled_transactions_april_sept.csv',
    index=False
)

print(f"Saved: labeled_transactions_april_sept.csv")
print(f"Shape: {sales_clean[output_cols].shape}")
print("\nNext step: run 02_temporal_feature_engineering.ipynb")